# Stagnation detection in practice

## Stagnation detection on arrays

We will use matrix multiplication as an example of other ways stagnation can be measured instead of using `report` on scalars.

### Matrix multiplication

Regular matrix multiplication in Julia can be performed as follows

In [ ]:
import Random: Xoshiro
rng = Xoshiro(1)

n = 10
m = 10

A = rand(rng, n, m)
B = rand(rng, m, n)

C = A * B

Modify matrix multiplication to perform stagnation detection and show the example functions like net_absorptions

The random number generator `rand` also supports type arguments, which we have extended to support stagnation detection types by conversion.

In [ ]:
using StagnationDetection

A = rand(rng, Float32SD, n, m)
B = rand(rng, Float32SD, m, n)

Multiplication `A*B` returns an $n \times n$ matrix where each entry is the sum of $m$ floating-point numbers, each being the product of two entries from the arguments.

In [ ]:
C = A*B

Int.(getfield.(C, :additions))

> A sidenote: the way operations are counted in matrix multiplication changes in different versions of Julia. The current version at the time of writing is $1.12$, where matrix multiplication starts every scalar product with a zero element and then accumulates $m$ values, therefore counting $m$ additions. If the accumulator is set to the first element instead, it would count $m-1$ additions.

We can perform large rectangular matrix multiplication with stagnation detection.

In [ ]:
n = 5
m = 1_000_000

rng = Xoshiro(1)

A = rand(rng, Float32SD, n, m) # uniform distribution 
B = randn(rng, Float32SD, m, n) # normal distribution

C = A * B

We have provided functions to report the number of absorptions taking place. First, we can show the overall stagnation including both rounding up and down.

In [ ]:
num_absorptions.(C)

But we may be more interested in the difference, assuming that absorptions in alternate rounding directions effectively cancel out.

In [ ]:
net_absorptions.(C)

The count for overall absorptions after cancellation is much lower.

We have a function `reportall` which behaves similarly to `report`. However we can use `reportall` to identify every variable with stagnation detection active within the global scope, considering both scalars and arrays, and provide short summaries for each. 

In [ ]:
reportall()

> Another sidenote: stagnation detection does not work for repeated matrix multiplication. This is because scalar operations other than addition discard stagnation information, so the stagnation data for each matrix element will be discarded when a subsequent matrix multiplication is performed.

The matrices `A` and `B` are detected due to their type, but are directly constructed and thus have no stagnation history. The matrix `C` reports the minimum, maximum and mean quantities `diff_absorptions/additions` as percentages for all elements in the matrix. There are no scalars in the current scope, so none are shown.

While we used `net_absorptions` in the example above to show mixed sign in the stagnation counts, `reportall` uses `diff_absorptions` to identify stagnation without regard to sign.

For dense matrix multiplication, each element in the product will have undergone the same number of additions.